### 01 data prep + split
reads: 6 csv files in Dokumenty  
writes: prep/

#### setup

In [1]:
import os, json, numpy as np, pandas as pd

DATA = 'C:/Users/Nimishavarma/OneDrive/Dokumenty'
OUT  = os.path.join(DATA, 'hti-project', 'prep')
os.makedirs(OUT, exist_ok=True)
SEED = 42
rng = np.random.default_rng(SEED)

#### load csvs

In [2]:
ecfp = pd.read_csv(f'{DATA}/compound_features_ecfp4.csv')
aac  = pd.read_csv(f'{DATA}/target_features_aac.csv')
ct_pos = pd.read_csv(f'{DATA}/compound_target_labeled.csv', usecols=['compound_id','protein_id'])
ct_un  = pd.read_csv(f'{DATA}/compound_target_unlabeled.csv', usecols=['compound_id','protein_id'])
hc = pd.read_csv(f'{DATA}/herb_compound_edges.csv', sep='\t', usecols=['herb_id','compound_id'])
hct = pd.read_csv(f'{DATA}/herb_compound_target.csv', usecols=['herb_id','protein_id'])

hc = hc.drop_duplicates()
print(len(ecfp), len(aac), len(ct_pos), len(ct_un), len(hc), len(hct))

2319 1341 14166 3095613 5005 126830


#### integer index for every herb, compound, target

In [3]:
herbs = sorted(set(hc['herb_id']))
cmps  = sorted(set(ecfp['compound_id']))
tgts  = sorted(set(aac['protein_id']))

h2i = {x:i for i,x in enumerate(herbs)}
c2i = {x:i for i,x in enumerate(cmps)}
t2i = {x:i for i,x in enumerate(tgts)}

n_h, n_c, n_t = len(herbs), len(cmps), len(tgts)
print(n_h, n_c, n_t)

62 2319 1341


#### node features (ecfp4 for compounds, aac for targets)

In [4]:
ecfp = ecfp.set_index('compound_id').loc[cmps]
bit_cols = [c for c in ecfp.columns if c.startswith('ecfp4_')]
C_feat = ecfp[bit_cols].to_numpy(dtype=np.float32)

aac = aac.set_index('protein_id').loc[tgts]
aac_cols = [c for c in aac.columns if c.startswith('aac_')]
T_feat = aac[aac_cols].to_numpy(dtype=np.float32)

print(C_feat.shape, T_feat.shape)

(2319, 2048) (1341, 20)


#### edge arrays (row 0 = source, row 1 = destination)

In [5]:
hc_src = hc['herb_id'].map(h2i).to_numpy()
hc_dst = hc['compound_id'].map(c2i).to_numpy()
hc_edges = np.stack([hc_src, hc_dst])

ct_src = ct_pos['compound_id'].map(c2i).to_numpy()
ct_dst = ct_pos['protein_id'].map(t2i).to_numpy()
ct_all = np.stack([ct_src, ct_dst])

print(hc_edges.shape, ct_all.shape)

(2, 5005) (2, 14166)


#### split 1: edge hiding, C-T edges 70/15/15

In [6]:
N = ct_all.shape[1]
perm = rng.permutation(N)
n_tr = int(0.70 * N)
n_va = int(0.15 * N)

tr_idx = perm[:n_tr]
va_idx = perm[n_tr:n_tr+n_va]
te_idx = perm[n_tr+n_va:]

ct_train = ct_all[:, tr_idx]
ct_val_pos = ct_all[:, va_idx]
ct_test_pos = ct_all[:, te_idx]

print(ct_train.shape, ct_val_pos.shape, ct_test_pos.shape)

(2, 9916) (2, 2124) (2, 2126)


#### edge hiding negatives from unlabeled pool (val 1:10, test 1:50)

In [7]:
ct_un['c'] = ct_un['compound_id'].map(c2i)
ct_un['t'] = ct_un['protein_id'].map(t2i)
ct_un = ct_un.dropna(subset=['c','t']).astype({'c':'int64','t':'int64'})

pool = ct_un[['c','t']].to_numpy()

n_val_neg  = 10 * ct_val_pos.shape[1]
n_test_neg = 50 * ct_test_pos.shape[1]

pick = rng.choice(len(pool), size=n_val_neg + n_test_neg, replace=False)
val_neg_rows  = pool[pick[:n_val_neg]]
test_neg_rows = pool[pick[n_val_neg:]]

ct_val_neg  = val_neg_rows.T
ct_test_neg = test_neg_rows.T

print(ct_val_neg.shape, ct_test_neg.shape)

(2, 21240) (2, 106300)


#### herb-target pairs for ht eval

In [8]:
ht = hct.drop_duplicates(['herb_id','protein_id']).copy()
ht['h'] = ht['herb_id'].map(h2i)
ht['t'] = ht['protein_id'].map(t2i)
ht = ht.dropna(subset=['h','t']).astype({'h':'int64','t':'int64'})
ht_pairs = ht[['h','t']].to_numpy()
print(ht_pairs.shape)

(24835, 2)


#### split 2: compound disjoint, compounds 70/15/15

In [9]:
# compound disjoint split
ct_compounds = np.unique(ct_all[0])
rng.shuffle(ct_compounds)
nc = len(ct_compounds)
ntr_c = int(0.70 * nc)
nva_c = int(0.15 * nc)

tr_set = set(ct_compounds[:ntr_c].tolist())
va_set = set(ct_compounds[ntr_c:ntr_c+nva_c].tolist())
te_set = set(ct_compounds[ntr_c+nva_c:].tolist())

tr_m = np.isin(ct_all[0], list(tr_set))
va_m = np.isin(ct_all[0], list(va_set))
te_m = np.isin(ct_all[0], list(te_set))

ct_train_cd = ct_all[:, tr_m]
ct_vp_cd = ct_all[:, va_m]
ct_tp_cd = ct_all[:, te_m]

print('compound disjoint split')
print('train edges:', ct_train_cd.shape[1], 'from', len(tr_set), 'compounds')
print('val edges:', ct_vp_cd.shape[1], 'from', len(va_set), 'compounds')
print('test edges:', ct_tp_cd.shape[1], 'from', len(te_set), 'compounds')

compound disjoint split
train edges: 9750 from 508 compounds
val edges: 2172 from 108 compounds
test edges: 2244 from 110 compounds


#### compound disjoint negatives (only from val/test compounds)

In [10]:
# cd negatives from unlabeled pool (restricted to val/test compounds)
va_pool_cd = pool[np.isin(pool[:, 0], list(va_set))]
te_pool_cd = pool[np.isin(pool[:, 0], list(te_set))]

need_vn = 10 * ct_vp_cd.shape[1]
need_tn = 50 * ct_tp_cd.shape[1]

pv = rng.choice(len(va_pool_cd), size=min(need_vn, len(va_pool_cd)), replace=False)
if need_tn <= len(te_pool_cd):
    pt = rng.choice(len(te_pool_cd), size=need_tn, replace=False)
else:
    pt = rng.choice(len(te_pool_cd), size=need_tn, replace=True)

ct_vn_cd = va_pool_cd[pv].T
ct_tn_cd = te_pool_cd[pt].T

print('cd val neg:', ct_vn_cd.shape, 'test neg:', ct_tn_cd.shape)

cd val neg: (2, 21720) test neg: (2, 112200)


#### save

In [11]:
np.save(f'{OUT}/C_feat.npy', C_feat)
np.save(f'{OUT}/T_feat.npy', T_feat)
np.save(f'{OUT}/hc_edges.npy', hc_edges)

# edge hiding split
np.save(f'{OUT}/ct_train.npy', ct_train)
np.save(f'{OUT}/ct_val_pos.npy', ct_val_pos)
np.save(f'{OUT}/ct_val_neg.npy', ct_val_neg)
np.save(f'{OUT}/ct_test_pos.npy', ct_test_pos)
np.save(f'{OUT}/ct_test_neg.npy', ct_test_neg)

# compound disjoint split
np.save(f'{OUT}/ct_train_cd.npy', ct_train_cd)
np.save(f'{OUT}/ct_val_pos_cd.npy', ct_vp_cd)
np.save(f'{OUT}/ct_val_neg_cd.npy', ct_vn_cd)
np.save(f'{OUT}/ct_test_pos_cd.npy', ct_tp_cd)
np.save(f'{OUT}/ct_test_neg_cd.npy', ct_tn_cd)

np.save(f'{OUT}/ht_pairs.npy', ht_pairs)

with open(f'{OUT}/meta.json','w') as f:
    json.dump({'n_h': n_h, 'n_c': n_c, 'n_t': n_t, 'seed': SEED}, f)

with open(f'{OUT}/id_maps.json','w') as f:
    json.dump({'h2i': h2i, 'c2i': c2i, 't2i': t2i}, f)

print('saved to', OUT)

saved to C:/Users/Nimishavarma/OneDrive/Dokumenty\hti-project\prep
